# Objects-only ablation: H1.2 + Gate + visual cache
Attach original YOLO detection JSON (raw object class names), MSCOCO/split, and all 3 visual-cache HDF5 files. Set SOURCE and the object/name fields to the actual JSON schema. Old prompt embeddings alone are insufficient. No VLM and no rerun of YOLO. The notebook builds a fresh objects-only CLIP prompt cache, trains from scratch for 10 epochs, then tests all 5000 images.


In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = Path('/kaggle/working/Image_Captioning')
def run(args):
    args = list(map(str, args))
    print('Running:', ' '.join(args), flush=True)
    subprocess.run(args, check=True)
if not REPO.exists():
    run(['git', 'clone', 'https://github.com/Supzxjee/Image_Captioning.git', REPO])
os.chdir(REPO)
run(['git', 'fetch', 'origin'])
run(['git', 'checkout', 'main'])
run(['git', 'pull', '--ff-only'])
run(['git', 'rev-parse', 'HEAD'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])


In [ ]:
SOURCE = Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
OBJECTS_FIELD = 'objects'  # e.g. 'detections' if your JSON uses that key
NAME_KEY = 'label'  # Original saved YOLO class-name field
JSON = Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
IMAGES = Path('/kaggle/input/datasets/vuthetam/mscoco-2014/images')
CACHE = Path('/kaggle/input/datasets/ducanh2403/visual-cache')
PROMPT = Path('/kaggle/working/object_prompt_cache/prompt_yolo_objects.pt')
for path in (SOURCE, JSON):
    assert path.is_file(), f'Missing Input or wrong path: {path}'
assert IMAGES.is_dir(), IMAGES
for part in range(1, 4):
    assert (CACHE / f'visual_part_{part:02d}_of_03.h5').is_file(), CACHE
# Check the schema before expensive encoding/training.
import json
from captioning.object_prompts import object_names
source = json.loads(SOURCE.read_text(encoding='utf-8'))
if 'data' in source and isinstance(source['data'], dict):
    source = source['data']
first_key = next(iter(source))
print('Sample source:', first_key, source[first_key])
print('Extracted labels:', object_names(source[first_key], OBJECTS_FIELD, NAME_KEY))
del source


In [ ]:
run([sys.executable, '-u', 'build_object_prompts.py',
     '--source', SOURCE, '--objects-field', OBJECTS_FIELD, '--name-key', NAME_KEY,
     '--dataset-json-path', JSON, '--output', PROMPT])
# Cache .pt and .json are saved under /kaggle/working/object_prompt_cache.


In [ ]:
common = ['--dataset-json-path', JSON, '--base-path', IMAGES,
          '--prompt-cache-path', PROMPT, '--visual-cache', CACHE,
          '--visual-cache-id-key', 'coco_id', '--visual-preprocessing', 'bilinear',
          '--visual-precision', 'fp32']
VERIFY_VISUAL_CACHE = False  # Already verified in the previous own-cache experiment.
if VERIFY_VISUAL_CACHE:
    run([sys.executable, '-u', 'train_h1_2_gated.py', '--mode', 'verify-cache',
         '--split', 'val', '--limit', 10, *common])
run([sys.executable, '-u', 'train_h1_2_gated.py', '--mode', 'train',
     '--epochs', 10, '--seed', 42, '--batch-size', 32, '--num-workers', 0,
     '--experiment-name', 'h1_2_gated_yolo_objects_only', '--test-after-train', *common])
